# AI Trading System - Demo Notebook

This notebook demonstrates the key features of the AI Trading System.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import system modules
from src.risk.position_sizing import (
    FixedFractionalSizer, KellySizer, VolatilityTargetSizer, CPPISizer
)
from src.risk.monte_carlo import MonteCarloSimulator
from src.signal.strategies import MomentumStrategy, MeanReversionStrategy
from src.backtest.engine import BacktestEngine, BacktestConfig
from src.common.utils import sharpe_ratio, max_drawdown

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## 1. Position Sizing Demo

In [ ]:
# Create sample signal and portfolio
from src.common.types import Signal, PortfolioState, SignalType
from datetime import datetime

signal = Signal(
    symbol='AAPL',
    timestamp=datetime.utcnow(),
    signal_type=SignalType.BUY,
    confidence=0.75,
    expected_return=2.5,
    half_life_seconds=3600,
    strategy_id='momentum',
    suggested_stop=170.0,
    suggested_target=185.0
)

portfolio = PortfolioState(
    timestamp=datetime.utcnow(),
    equity=100000,
    cash=50000,
    positions={},
    trades=[]
)

current_price = 175.5
stop_price = 170.0

In [ ]:
# Compare different position sizing methods
sizers = {
    'Fixed Fractional (1%)': FixedFractionalSizer(risk_fraction=0.01),
    'Kelly (Quarter)': KellySizer(win_probability=0.52, win_loss_ratio=2.1, kelly_fraction=0.25),
    'Vol Target (10%)': VolatilityTargetSizer(target_volatility=0.10),
    'CPPI': CPPISizer(floor_fraction=0.90, multiplier=3.0)
}

print("Position Sizing Comparison\n" + "="*50)
for name, sizer in sizers.items():
    result = sizer.calculate_size(signal, portfolio, current_price, stop_price)
    print(f"\n{name}:")
    print(f"  Quantity: {result.quantity:.0f}")
    print(f"  Dollar Risk: ${result.dollar_risk:,.2f}")
    print(f"  Position Value: ${result.position_value:,.2f}")
    print(f"  Leverage: {result.leverage:.2%}")
    print(f"  Risk Fraction: {result.risk_fraction:.2%}")

## 2. Monte Carlo Simulation Demo

In [ ]:
# Run Monte Carlo simulation
sim = MonteCarloSimulator(seed=42)

results = sim.simulate_fixed_parameters(
    n_trades=1000,
    n_sims=20000,
    win_probability=0.52,
    win_loss_ratio=2.1,
    risk_per_trade=0.01,
    starting_equity=100000,
    ruin_threshold=0.20
)

print(results.summary())

In [ ]:
# Visualize Monte Carlo results
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Terminal wealth distribution
terminal_wealths = [curve[-1] for curve in results.equity_curves[:1000]]
axes[0, 0].hist(terminal_wealths, bins=50, alpha=0.7, edgecolor='black')
axes[0, 0].axvline(results.terminal_wealth_mean, color='red', linestyle='--', label=f'Mean: ${results.terminal_wealth_mean:,.0f}')
axes[0, 0].axvline(results.terminal_wealth_median, color='green', linestyle='--', label=f'Median: ${results.terminal_wealth_median:,.0f}')
axes[0, 0].set_xlabel('Terminal Wealth')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Terminal Wealth Distribution')
axes[0, 0].legend()

# Max drawdown distribution
axes[0, 1].hist(results.max_drawdowns, bins=50, alpha=0.7, edgecolor='black', color='orange')
axes[0, 1].axvline(results.max_drawdown_mean, color='red', linestyle='--', label=f'Mean: {results.max_drawdown_mean:.2%}')
axes[0, 1].set_xlabel('Max Drawdown')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Max Drawdown Distribution')
axes[0, 1].legend()

# Sample equity curves
for curve in results.equity_curves[:100]:
    axes[1, 0].plot(curve, alpha=0.1, color='blue')
axes[1, 0].set_xlabel('Trade Number')
axes[1, 0].set_ylabel('Equity')
axes[1, 0].set_title('Sample Equity Curves (n=100)')

# Sample drawdown curves
for dd_curve in results.drawdown_curves[:100]:
    axes[1, 1].plot(dd_curve, alpha=0.1, color='red')
axes[1, 1].set_xlabel('Trade Number')
axes[1, 1].set_ylabel('Drawdown')
axes[1, 1].set_title('Sample Drawdown Curves (n=100)')

plt.tight_layout()
plt.show()

## 3. Kelly Analysis

In [ ]:
# Analyze Kelly criterion
kelly_results = sim.kelly_analysis(
    win_probability=0.52,
    win_loss_ratio=2.1,
    risk_fractions=np.linspace(0, 0.5, 51)
)

# Plot Kelly analysis
plt.figure(figsize=(12, 6))

plt.plot(kelly_results['risk_fraction'], kelly_results['geometric_growth_rate'], linewidth=2)
plt.axvline(kelly_results[kelly_results['is_full_kelly']]['risk_fraction'].values[0], 
           color='red', linestyle='--', label='Full Kelly')
plt.axvline(kelly_results[kelly_results['is_half_kelly']]['risk_fraction'].values[0], 
           color='orange', linestyle='--', label='Half Kelly')
plt.axvline(kelly_results[kelly_results['is_quarter_kelly']]['risk_fraction'].values[0], 
           color='green', linestyle='--', label='Quarter Kelly')

plt.xlabel('Risk Fraction per Trade')
plt.ylabel('Geometric Growth Rate')
plt.title('Kelly Criterion Analysis')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 4. Strategy Demo

In [ ]:
# Generate sample price data
np.random.seed(42)
n_days = 252
dates = pd.date_range(start='2025-01-01', periods=n_days, freq='D')

# Generate random walk with trend
returns = np.random.normal(0.0005, 0.02, n_days)
prices = 100 * np.exp(np.cumsum(returns))

# Create OHLCV data
data = pd.DataFrame({
    'open': prices * (1 + np.random.normal(0, 0.001, n_days)),
    'high': prices * (1 + np.abs(np.random.normal(0, 0.01, n_days))),
    'low': prices * (1 - np.abs(np.random.normal(0, 0.01, n_days))),
    'close': prices,
    'volume': np.random.randint(1000000, 10000000, n_days)
}, index=dates)

# Ensure high >= low
data['high'] = np.maximum(data['high'], data[['open', 'close']].max(axis=1))
data['low'] = np.minimum(data['low'], data[['open', 'close']].min(axis=1))

print(f"Generated {len(data)} days of price data")
data.head()

In [ ]:
# Test momentum strategy
momentum_strategy = MomentumStrategy(lookback_periods=[20, 60], adx_threshold=25)

# Generate signals
signals = []
for i in range(100, len(data)):
    signal = momentum_strategy.generate_signal(data.iloc[:i], 'AAPL')
    if signal:
        signals.append({
            'date': data.index[i],
            'signal_type': signal.signal_type.value,
            'confidence': signal.confidence,
            'price': data['close'].iloc[i]
        })

signals_df = pd.DataFrame(signals)
print(f"Generated {len(signals_df)} signals")
signals_df.head(10)

In [ ]:
# Visualize signals on price chart
plt.figure(figsize=(14, 7))

plt.plot(data.index, data['close'], label='Price', alpha=0.7)

if len(signals_df) > 0:
    buy_signals = signals_df[signals_df['signal_type'] == 'buy']
    sell_signals = signals_df[signals_df['signal_type'] == 'sell']
    
    plt.scatter(buy_signals['date'], buy_signals['price'], 
               color='green', marker='^', s=100, label='Buy Signal', zorder=5)
    plt.scatter(sell_signals['date'], sell_signals['price'], 
               color='red', marker='v', s=100, label='Sell Signal', zorder=5)

plt.xlabel('Date')
plt.ylabel('Price')
plt.title('Momentum Strategy Signals')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 5. Summary

This demo notebook showcased:

1. **Position Sizing**: Multiple sizing algorithms (Fixed Fractional, Kelly, Volatility Targeting, CPPI)
2. **Monte Carlo Simulation**: Risk of ruin analysis with equity curve simulation
3. **Kelly Analysis**: Optimal bet sizing based on win probability and payoff ratio
4. **Strategy Signals**: Momentum strategy signal generation and visualization

For more details, see the full documentation in the `docs/` directory.